# Web of Life — Plant-Pollinator Network Analysis

Structural comparison of **158 plant-pollinator networks** from the [Web of Life](https://www.web-of-life.es/) database (174 loaded; 16 excluded for having fewer than 20 nodes).

Networks span mainland sites, continental islands and oceanic islands worldwide, following the geographical classification of Traveset et al. (2015, *Global Ecology and Biogeography*).

## Pipeline

| Step | Description |
|------|-------------|
| **1. Graph construction** | Each CSV (plants × pollinators interaction matrix) is parsed into a weighted bipartite NetworkX graph. |
| **2. Embedding** | Every network is embedded with `node_embedding` (EDRep) at a fixed dimension `DIM = 16`, regardless of network size. |
| **3. Distance matrices** | Three 158 × 158 pairwise distance matrices are built from Wasserstein-2 comparisons of scalar-product distributions: *unweighted class*, *weighted class *, *whole-graph*. |
| **4. Statistical tests** | PERMANOVA (island type), Mantel (latitude) and ANOSIM (biogeographic region) are applied to each distance matrix. |
| **5. Comparison** | The three distances are benchmarked against each other on all three tests. |

## Imports and setup

In [ ]:
import sys, os, glob, time
sys.path.insert(0, r"C:\Users\Utente\Desktop\progetto vscode\src")

import numpy as np
import pandas as pd
import networkx as nx
import ot
import matplotlib.pyplot as plt
import seaborn as sns
import json as _json, pathlib

from functions import *

from skbio.stats.distance import DistanceMatrix, permanova, mantel, anosim
from scipy.spatial.distance import pdist, squareform

## 1. Bipartite Graph Construction

`create_bipartite_graph` reads a plants × pollinators interaction matrix from CSV and builds a weighted bipartite `networkx` graph:

- Rows are **plant species** (`bipartite=0`), columns are **pollinator species** (`bipartite=1`).
- Isolated species (zero rows or columns) are retained as nodes to preserve the full species list.
- Edges are added only where the interaction weight is **strictly positive**, using `df.stack()` filtered to `> 0`. This avoids an O(P × I) double loop and correctly handles the change in `dropna` default behaviour introduced in pandas ≥ 2.2.

The function also returns the integer indices of plant and pollinator nodes within the node list, which are needed downstream to split the embedding into the two partitions.

In [ ]:
def create_bipartite_graph(file_path):
    df = pd.read_csv(file_path, index_col=0, sep=',')
    df = df.apply(pd.to_numeric, errors='coerce').fillna(0)

    plants = df.index.tolist()
    pollinators = df.columns.tolist()

    G = nx.Graph()
    G.add_nodes_from(plants, bipartite=0)
    G.add_nodes_from(pollinators, bipartite=1)

    # stack() on the df AFTER fillna(0): no NaN in the df, so the result is clean.
    # df[df > 0].stack() masked zeros as NaN and in pandas >= 2.2
    # the dropna default changed: NaN values ended up as edge weights (nan > 0 is False).
    s = df.stack()
    s = s[s > 0]
    G.add_weighted_edges_from((p, i, float(w)) for (p, i), w in s.items())

    nodes = list(G.nodes())
    partition_plant = [k for k, n in enumerate(nodes) if G.nodes[n]['bipartite'] == 0]
    partition_pollinators = [k for k, n in enumerate(nodes) if G.nodes[n]['bipartite'] == 1]

    return G, partition_plant, partition_pollinators, plants, pollinators

## 2. Network Loading and Embedding

All 174 CSV files are loaded and embedded in a single pass. Each network is identified by its Web of Life ID (`M_PL_xxx`). For each network, `node_embedding` (EDRep) is called with the plant and pollinator index lists, learning a `d`-dimensional representation of every node by minimising a variational softmax loss over random-walk co-occurrence statistics.

**Why a fixed dimension?** `DIM = 16` is used for all networks regardless of size. An adaptive dimension (e.g. `n/5` or `n/10`) would make the embedding geometry change with `n`, introducing a spurious size-dependent variable and causing distances between large and small networks to be inhomogeneous.

After loading, a **size filter** is applied: networks with fewer than `MIN_NODES = 20` nodes are discarded, leaving **158 networks** for all subsequent steps. The 16 excluded networks are listed in the cell output below.

In [ ]:
folder = r"C:\Users\Utente\Desktop\progetto vscode\data\data_wol\file_bipatite"
path_files = sorted(glob.glob(os.path.join(folder, "*.csv")))
print(f"{len(path_files)} networks found")

graphs = {}        # id -> graph
embeddings = {}    # id -> X_list ([X_plants, X_pollinators])
graph_ids = []     # processing order
DIM = 16
t0 = time.time()
for f in path_files:
    gid = os.path.basename(f).replace('.csv', '')
    g, ppl, ppo, plants, pollinators = create_bipartite_graph(f)
    g.graph['name'] = gid
    X_list = node_embedding(g, [ppl, ppo], DIM)
    graphs[gid] = g
    embeddings[gid] = X_list
    graph_ids.append(gid)
print(f"Embedding of {len(graph_ids)} networks in {time.time() - t0:.1f}s")

In [ ]:
# Remove networks with fewer than 15 nodes from the entire analysis
MIN_NODES = 20
n_before = len(graph_ids)
excluded = {gid for gid in graph_ids if len(graphs[gid].nodes()) < MIN_NODES}

graph_ids  = [gid for gid in graph_ids if gid not in excluded]
graphs     = {gid: g for gid, g in graphs.items()     if gid not in excluded}
embeddings = {gid: e for gid, e in embeddings.items() if gid not in excluded}

print(f"Threshold: ≥ {MIN_NODES} nodes  |  networks kept: {len(graph_ids)} / {n_before}")
if excluded:
    print(f"Excluded ({len(excluded)}): " + ", ".join(sorted(excluded)))

## 3. Metadata: Island Type and Biogeographic Region

Two grouping variables are attached to each network from `references.csv`:

**Island type** — manually assigned for each of the 94 distinct localities in the dataset, following the geological classification used by Traveset et al. (2015):
- `oceanic island` — formed over oceanic plates, never connected to a continental landmass (e.g. Galápagos, Canary Islands, Mauritius).
- `continental island` — ancient continental fragments or recent continental shelf islands (e.g. Japan, New Zealand, Greenland).
- `mainland` — continental sites (default fallback for unrecognised localities).

**Biogeographic region** — automatically derived from latitude and longitude via `realm_from_latlon`, assigning each network to one of five coarse realms: Nearctic, Neotropical, Palearctic, Afrotropical or Australasian.


In [ ]:
def realm_from_latlon(lat, lon):
    """Approximate biogeographic realm from coordinates (coarse bounding boxes)."""
    if pd.isna(lat) or pd.isna(lon):
        return np.nan
    lat, lon = float(lat), float(lon)
    # Americas
    if -170 <= lon <= -34:
        return 'Nearctic' if lat >= 13 else 'Neotropical'
    # Australasia (Australia / New Zealand / New Guinea)
    if lon >= 110 and lat <= -10:
        return 'Australasian'
    # Indomalayan (South / South-East Asia)
    if 60 <= lon <= 150 and -10 <= lat <= 28:
        return 'Indomalayan'
    # Afrotropical (sub-Saharan Africa, Arabia, Indian Ocean islands)
    if -20 <= lon <= 60 and lat <= 21:
        return 'Afrotropical'
    # Rest of Eurasia / North Africa
    return 'Palearctic'

In [ ]:
island_mapping = {
    # Continental Island 
    'Amami-Ohsima Island, Japan': 'continental island',
    'Arima Valley': 'continental island',
    "Arthur's Pass, New Zealand": 'continental island',
    'Ashu, Kyoto, Japan': 'continental island',
    'Bristol, England': 'continental island',
    'Cass, New Zealand': 'continental island',
    'Chiloe, Chile': 'continental island',
    'Craigieburn, New Zealand': 'continental island',
    'Hazen Camp, Ellesmere Island, Canada': 'continental island',
    'Hickling, Norfolk, UK': 'continental island',
    'Kibune, Kyoto, Japan': 'continental island',
    'Kyoto City, Japan': 'continental island',
    'Matamata': 'continental island',
    'Melville Island, Canada': 'continental island',
    'Morne Seychellois National Park, Mahé': 'continental island',
    'Mt. Kushigata, Yamanashi Pref., Japan': 'continental island',
    'Mt. Yufu, Japan': 'continental island',
    'Nakaikemi marsh, Fukui Prefecture, Japan': 'continental island',
    'Shelfanger, Norfolk, UK': 'continental island',
    'Tundra, Greenladn': 'continental island',
    'Uummannaq Island, Greenland': 'continental island',
    'Zackenberg': 'continental island',
    
    # Oceanic Island 
    'Black River Gorges National Park, Mauritius': 'oceanic island',
    'Flores, Açores': 'oceanic island',
    'Galapagos': 'oceanic island',
    'Garajonay, Gomera, Spain': 'oceanic island',
    'Mauritius Island': 'oceanic island',
    'Morant Point, Jamaica': 'oceanic island',
    'Puerto Villamil, Isabela Island, Galapagos': 'oceanic island',
    'Syndicate, Dominica': 'oceanic island',
    'Tenerife, Canary Islands': 'oceanic island',
    'Windsor, The Cockpit Country, Jamaica': 'oceanic island'
}

In [ ]:
ref_path = r"C:\Users\Utente\Desktop\progetto vscode\data\data_wol\reference\references.csv"
metadata = pd.read_csv(ref_path, encoding='latin-1')
metadata['ID'] = metadata['ID'].astype(str)
metadata = metadata.set_index('ID')

# robust alignment to the processed graphs (no off-by-one)
metadata = metadata.reindex(graph_ids)

metadata['Latitude'] = pd.to_numeric(metadata['Latitude'], errors='coerce')
metadata['Longitude'] = pd.to_numeric(metadata['Longitude'], errors='coerce')

# fillna('mainland') as fallback for unrecognised localities (e.g. encoding edge-case)
metadata['island_type'] = metadata['Locality of Study'].map(island_mapping).fillna('mainland')
metadata['region'] = metadata.apply(
    lambda r: realm_from_latlon(r['Latitude'], r['Longitude']), axis=1)

print(metadata[['Locality of Study', 'island_type', 'region']].head(10))
print('\nNetworks by island type:')
print(metadata['island_type'].value_counts(dropna=False))
print('\nNetworks by region:')
print(metadata['region'].value_counts(dropna=False))

## 4. Class-Wise Wasserstein Distance Matrix (Unweighted — Base)

For a bipartite network with plant nodes $P$ and pollinator nodes $I$, the EDRep embedding produces two matrices: $X_P \in \mathbb{R}^{|P| \times d}$ and $X_I \in \mathbb{R}^{|I| \times d}$. The structural distance between two networks A and B is computed by comparing the distributions of pairwise scalar products across **three bipartite blocks**:

| Block | Scalar products | Pairs used |
|-------|----------------|------------|
| (0, 0) plant–plant | $X_P^A (X_P^A)^\top$ vs $X_P^B (X_P^B)^\top$ | upper triangle (k=1) |
| (0, 1) plant–pollinator | $X_P^A (X_I^A)^\top$ vs $X_P^B (X_I^B)^\top$ | all entries |
| (1, 1) pollinator–pollinator | $X_I^A (X_I^A)^\top$ vs $X_I^B (X_I^B)^\top$ | upper triangle (k=1) |

Within each block the 1D Wasserstein-2 distance $W_{2,k}$ is computed between the two sorted value arrays. The base distance aggregates block distances with **equal weights** (Euclidean norm):

$$D_{\text{base}}(A, B) = \sqrt{\sum_k W_{2,k}^2}$$

**Implementation note:** block distributions are precomputed once per network (`block_dists`) and sorted; only the 1D Wasserstein calls remain in the O(n²) pair loop. The resulting 158 × 158 matrix is saved to `dist_main.npy`.

In [ ]:
CACHE_DIR = pathlib.Path(r"C:\Users\Utente\Desktop\progetto vscode\data\data_wol\distances")
CACHE_DIR.mkdir(exist_ok=True)
_ids_file = CACHE_DIR / "graph_ids.json"

def _cache_valid(*mat_files):
    """True if all .npy files exist and the saved graph_ids match the current list."""
    if not _ids_file.exists() or not all(p.exists() for p in mat_files):
        return False
    with open(_ids_file) as _f:
        return _json.load(_f) == graph_ids

def _save_ids():
    with open(_ids_file, "w") as _f:
        _json.dump(graph_ids, _f)

print(f"Cache directory: {CACHE_DIR}")

### Cache directory setup

Distance matrices are expensive to recompute (158 × 157 / 2 = 12,403 pairs per matrix). Results are cached as `.npy` files under `data/data_wol/distances/`. `_cache_valid` checks that both the files exist and the saved `graph_ids` list matches the current one — a mismatch (e.g. after adding or removing networks) triggers a full recompute.

In [ ]:
precomp = {gid: block_dists(embeddings[gid]) for gid in graph_ids}

_f_main = CACHE_DIR / "dist_main.npy"

if _cache_valid(_f_main):
    dist_matrix = np.load(_f_main)
    print(f"Loaded dist_matrix from cache  ({_f_main.name})")
else:
    n = len(graph_ids)
    dist_matrix = np.zeros((n, n))
    for a in range(n):
        for b in range(a + 1, n):
            d = graph_distance(precomp[graph_ids[a]], precomp[graph_ids[b]])
            dist_matrix[a, b] = dist_matrix[b, a] = d
    print("Distance matrix computed")
    np.save(_f_main, dist_matrix)
    _save_ids()
    print(f"Saved → {_f_main.name}")

dm = DistanceMatrix(dist_matrix, ids=graph_ids)

## 4b. Weighted Distance Matrix 
As a variant for comparison, the distance is also computed with **weights proportional to the number of pairs** in each block (`mode='n_pairs'`):

$$D_{\text{weighted}}(A, B) = \sqrt{\sum_k w_k\, W_{2,k}^2}, \qquad w_k = \frac{n_k}{\sum_j n_j}$$

where $n_k$ is the average number of scalar-product pairs in block $k$ between A and B. This is the **BLUE** (Best Linear Unbiased Estimator) under homogeneous estimation variance — up-weighting larger blocks reduces estimation error.

| Mode | Aggregation | Rationale |
|------|-------------|-----------|
| `none` (unweighted, **base**) | $\sqrt{\sum_k W_{2,k}^2}$ | All blocks contribute equally — canonical distance |
| `n_pairs` (weighted) | $\sqrt{\sum_k w_k\, W_{2,k}^2}$, $\;w_k \propto n_k$ | Blocks contribution proportional to size— down-weights noisy small blocks |

Both matrices are saved to disk (`dist_none.npy`, `dist_npairs.npy`) and compared against the whole-graph distance in Section 6.

In [ ]:
def graph_distance_weighted(bd_a, bd_b):
    """Weighted variant: block distances aggregated with weights proportional to number of pairs."""
    keys = [k for k in bd_a if k in bd_b]
    D, n_sizes = [], []
    for blk in keys:
        va, vb = bd_a[blk], bd_b[blk]
        D.append(ot.wasserstein_1d(va, vb, p=2) ** 0.5)
        n_sizes.append((len(va) + len(vb)) / 2)
    D = np.array(D)
    w = np.array(n_sizes)
    w = w / w.sum()
    return np.sqrt(np.dot(w, D**2))


_f_none   = CACHE_DIR / "dist_none.npy"
_f_npairs = CACHE_DIR / "dist_npairs.npy"

if _cache_valid(_f_none, _f_npairs):
    dist_matrices = {
        'none':    np.load(_f_none),
        'n_pairs': np.load(_f_npairs),
    }
    print(f"Loaded dist_matrices from cache  ({_f_none.name}, {_f_npairs.name})")
else:
    n_nets = len(graph_ids)
    dist_matrices = {}
    for mode, dist_fn in [('none', graph_distance), ('n_pairs', graph_distance_weighted)]:
        mat = np.zeros((n_nets, n_nets))
        t0 = time.time()
        for a in range(n_nets):
            for b in range(a + 1, n_nets):
                d = dist_fn(precomp[graph_ids[a]], precomp[graph_ids[b]])
                mat[a, b] = mat[b, a] = d
        dist_matrices[mode] = mat
        print(f"mode={mode!r:10s}  {time.time()-t0:.1f}s")
    np.save(_f_none,   dist_matrices['none'])
    np.save(_f_npairs, dist_matrices['n_pairs'])
    _save_ids()
    print(f"Saved → {_f_none.name}, {_f_npairs.name}")

dm_none = DistanceMatrix(dist_matrices['none'], ids=graph_ids)
print("dm_none created")

## 5. Statistical Tests on the Unweighted Class Distance

Three permutation-based tests are applied to the unweighted class distance matrix (`dm`, 158 × 158) to assess whether embedding-based structural dissimilarity is explained by ecological or geographical groupings:

| Test | Grouping variable | Question asked |
|------|-------------------|----------------|
| **PERMANOVA** | Island type (3 groups) | Do mainland, continental-island and oceanic-island networks have different structural compositions? |
| **Mantel** | Latitude (continuous) | Does latitudinal distance between sampling sites predict structural dissimilarity? |
| **ANOSIM** | Biogeographic region (5 groups) | Do networks from different biogeographic realms cluster structurally? |

All tests use 999 permutations. The `subset_dm` helper filters out networks with missing metadata before each test. Results for all three distance variants are compared in Section 6.

In [ ]:
def subset_dm(dm, series):
    """Sub-DistanceMatrix with only the non-NaN IDs in the series + aligned grouping."""
    valid = series.dropna()
    ids = [i for i in dm.ids if i in valid.index]
    return dm.filter(ids), valid.loc[ids].values, ids

### PERMANOVA — Island Type

PERMANOVA (Permutational Multivariate Analysis of Variance) partitions the total sum of squared distances into between-group and within-group components, producing a **pseudo-F statistic**. Significance is assessed by randomly permuting group labels (999 permutations). Here the three groups are: `mainland`, `continental island`, `oceanic island`.

In [ ]:
sub, grouping, ids = subset_dm(dm, metadata['island_type'])
print(f"PERMANOVA island_type on {len(ids)} networks")
print(permanova(sub, grouping, permutations=999))

### Mantel Test — Latitude

The Mantel test measures the Spearman rank correlation between two distance matrices and evaluates its significance via permutation (999 permutations). Here it tests whether **latitudinal distance** between sampling sites predicts **structural dissimilarity** between the corresponding networks.

In [ ]:
lat = metadata['Latitude'].dropna()
ids = [i for i in dm.ids if i in lat.index]
sub = dm.filter(ids)
lat_dist = squareform(pdist(lat.loc[ids].values.reshape(-1, 1), metric='euclidean'))
lat_dm = DistanceMatrix(lat_dist, ids=ids)
corr, p_val, _ = mantel(sub, lat_dm, method='spearman')
print(f"MANTEL latitude on {len(ids)} networks — R: {corr:.3f}, p-value: {p_val:.3f}")

### ANOSIM — Biogeographic Region

ANOSIM (Analysis of Similarities) tests whether between-group distances are systematically larger than within-group distances. The statistic R ranges from −1 to 1, where **R = 1** means complete separation of groups and **R = 0** means no separation. Significance is assessed via permutation (999 permutations). Here the five groups are the biogeographic realms derived from coordinates: Nearctic, Neotropical, Palearctic, Afrotropical and Australasian.

In [ ]:
sub, grouping, ids = subset_dm(dm, metadata['region'])
print(f"ANOSIM region on {len(ids)} networks")
print(anosim(sub, grouping, permutations=999))

---

## 6. Whole-Graph Wasserstein Distance and Final Comparison

The class-wise distances in Sections 4–5 exploit the plants/pollinators bipartition, computing Wasserstein distances separately for the three bipartite blocks (plant–plant, plant–pollinator, pollinator–pollinator). Here a third variant is introduced that **ignores the bipartition entirely**:

**Whole-graph distance** — each network is re-embedded with a single node class (all nodes together) using the same `DIM = 16`, and the W2 distance is computed between the scalar-product distributions of the full Gram matrices $X X^\top$.

Comparing this variant to the class distances allows us to quantify how much information the bipartite block structure contributes to discriminating networks. All three distances operate on the same set of **158 networks**.

In [ ]:
print(f"Re-embedding whole-graph of {len(graph_ids)} networks with dim={DIM}")

emb_global = {}
t0 = time.time()
for gid in graph_ids:
    g = graphs[gid]
    all_nodes = list(range(len(g.nodes())))
    emb_global[gid] = node_embedding(g, [all_nodes], DIM)[0]
print(f"Re-embedding completed in {time.time() - t0:.1f}s")

In [ ]:
def whole_features(X):
    """Whole-graph scalar-product distribution (for whole-graph W2)."""
    return np.sort((X @ X.T).flatten())

feat = {gid: whole_features(emb_global[gid]) for gid in graph_ids}

In [ ]:
_f_global = CACHE_DIR / "dist_global.npy"

if _cache_valid(_f_global):
    D_wass_global = np.load(_f_global)
    print(f"Loaded D_wass_global from cache  ({_f_global.name})")
else:
    n = len(graph_ids)
    D_wass_global = np.zeros((n, n))
    t0 = time.time()
    for a in range(n):
        Sa = feat[graph_ids[a]]
        for b in range(a + 1, n):
            Sb = feat[graph_ids[b]]
            w = ot.wasserstein_1d(Sa, Sb, p=2) ** 0.5
            D_wass_global[a, b] = D_wass_global[b, a] = w
    print(f"Matrix {n}x{n} computed in {time.time() - t0:.1f}s")
    np.save(_f_global, D_wass_global)
    _save_ids()
    print(f"Saved → {_f_global.name}")

dm_wass_global = DistanceMatrix(D_wass_global, ids=graph_ids)
assert np.allclose(D_wass_global, D_wass_global.T) and np.allclose(np.diag(D_wass_global), 0) and np.isfinite(D_wass_global).all()
print("Checks OK: symmetric, zero diagonal, finite")

### Pairwise Correlation Between the Three Distance Variants

Each scatter plot shows the 12,403 upper-triangle pairwise distances, comparing two variants at a time. Spearman and Pearson correlations are reported in the title.

The three distances are broadly consistent (all positively correlated), but capture different aspects of network structure: the whole-graph distance, which pools all nodes without regard to biological role, shows higher PERMANOVA pseudo-F and ANOSIM R than either class-based variant (see the test comparison below).

In [ ]:
from scipy.stats import spearmanr, pearsonr
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

sns.set_theme(style="whitegrid", palette="deep", font_scale=1.05)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (x, y, xlabel, ylabel) in zip(axes, pairs):
    rs, _ = spearmanr(x, y)
    rp, _ = pearsonr(x, y)

    ax.scatter(x, y,
               s=2,           # small dot size for dense scatterplots
               alpha=0.15,    # high transparency to reveal density
               color='#DD6722',
               edgecolors='none')

    sns.despine(ax=ax)
    ax.grid(True, linestyle='--', alpha=0.4)

    ax.set_xlabel(xlabel.capitalize(), fontsize=12, color="#333333")
    ax.set_ylabel(ylabel.capitalize(), fontsize=12, color='#333333')
    ax.set_title(f"Spearman = {rs:.3f}  |  Pearson = {rp:.3f}",
                 fontsize=13, fontweight='bold', pad=12)

plt.suptitle("Correlation between the three distances",
             fontsize=16, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

### Statistical Tests Across All Three Distance Variants

The same three tests (PERMANOVA on island type, Mantel on latitude, ANOSIM on region) are run on all three distance matrices via the generic `run_tests` function, producing a single comparison table.

In [ ]:
def run_tests(dmx):
    """Run PERMANOVA (island type), Mantel (latitude) and ANOSIM (region) on a DistanceMatrix."""
    sub, grp, _ = subset_dm(dmx, metadata['island_type'])
    perm = permanova(sub, grp, permutations=999)

    lat = metadata['Latitude'].dropna()
    ids_lat = [i for i in dmx.ids if i in lat.index]
    lat_dm = DistanceMatrix(squareform(pdist(lat.loc[ids_lat].values.reshape(-1, 1))), ids=ids_lat)
    r_mantel, p_mantel, _ = mantel(dmx.filter(ids_lat), lat_dm, method='spearman')

    sub_r, grp_r, _ = subset_dm(dmx, metadata['region'])
    ano = anosim(sub_r, grp_r, permutations=999)

    return {
        'PERMANOVA island (F)': perm['test statistic'],
        'PERMANOVA island (p)': perm['p-value'],
        'Mantel lat (R)':      r_mantel,
        'Mantel lat (p)':      p_mantel,
        'ANOSIM region (R)':  ano['test statistic'],
        'ANOSIM region (p)':  ano['p-value'],
    }

tabella = pd.DataFrame({
    'unweighted classes': run_tests(dm_none),
    'weighted classes': run_tests(dm),
    'whole-graph':       run_tests(dm_wass_global),
})
print("Statistical tests comparison:\n")
print(tabella.round(4))

---

## Conclusions


**Three consistent findings across all distance variants:**

1. **Island type drives network structure (PERMANOVA, p < 0.05 for all variants).** Oceanic-island, continental-island and mainland networks are structurally distinguishable using embedding-based distances alone, without computing any explicit ecological metric. This replicates, from a representation-learning perspective, the central result of Traveset et al. (2015).

2. **Latitude has no effect (Mantel, p > 0.20 for all variants).** Structural dissimilarity between networks is not predicted by latitudinal distance between their sampling sites. Local ecological context (species pool, climate, evolutionary history) dominates over broad geographic gradients.

3. **Biogeographic region is technically significant but ecologically negligible (ANOSIM R ≈ 0.05–0.09).** The weak effect reflects the fact that within-region variance is much larger than between-region variance; significance is partly an artefact of the large sample size (158 networks).

**On the role of the bipartition:** the whole-graph distance (which ignores the plant/pollinator distinction) is consistently the most sensitive discriminator. This suggests that the global spectral structure of the network — rather than the specific organisation of plant–plant, plant–pollinator or pollinator–pollinator interactions — carries the strongest signal for distinguishing mainland from island networks.